# DLB CSF Proteomics Pipeline

Reimplementation of del Campo et al. (2023, *Nature Communications*)  
"CSF proteome profiling reveals biomarkers to discriminate dementia with Lewy bodies from Alzheimer's disease"

**Target results:** 7-protein panel (DDC, FCER2, CRH, MMP-3, ABL1, MMP-10, THOP1)  
- DLB vs CN: AUC 0.947  
- DLB vs AD: AUC 0.929

## 1 · Setup

In [ ]:
# Clone the repo (replace with your GitHub URL if public, or skip if already mounted)
import os

REPO_URL = ""  # e.g. "https://github.com/yourname/lewy-body.git"
REPO_DIR = "/content/lewy_body"

if REPO_URL:
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Mount Google Drive and point to repo cloned there
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = "/content/drive/MyDrive/lewy_body"  # adjust path as needed

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
!pip install -q adjusttext tqdm statsmodels scikit-learn seaborn

import sys
sys.path.insert(0, "src")
print("src/ added to path")

In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lewy.data import load_discovery, load_validation, encode_labels, PANEL_PROTEINS
from lewy.features import filter_by_lod, differential_proteins
from lewy.model import build_classifier, repeated_stratified_cv, compute_auc_ci, select_panel
from lewy.evaluate import roc_curve_data, validate_on_cohort, format_metrics
from lewy.plots import plot_volcano, plot_roc, plot_violin, plot_forest

print("Imports OK")

## 2 · Configuration

In [ ]:
# ── Tune these ────────────────────────────────────────────────────────────────
N_REPEATS = 100       # 1000 for paper-faithful results (slower)
USE_CACHE = True      # skip CV if results/metrics/ JSON already exists
OUTPUT_DIR = Path("results")
# ─────────────────────────────────────────────────────────────────────────────

FIG_DIR = OUTPUT_DIR / "figures"
MET_DIR = OUTPUT_DIR / "metrics"
FIG_DIR.mkdir(parents=True, exist_ok=True)
MET_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output: {OUTPUT_DIR}  |  CV repeats: {N_REPEATS}")

## 3 · Load & Filter Data

In [ ]:
X, meta = load_discovery()
X = filter_by_lod(X, lod_frac=0.85)

print(f"Samples : {X.shape[0]}")
print(f"Proteins: {X.shape[1]} (after LOD filter)")
print(f"Labels  : {meta['Dx_group'].value_counts().to_dict()}")

## 4 · Differential Protein Analysis

In [ ]:
diff_dlb_cn = differential_proteins(X, meta, "CN", "DLB")
diff_dlb_ad = differential_proteins(X, meta, "AD", "DLB")

print(f"DLB vs CN: {(diff_dlb_cn['qval'] < 0.05).sum()} significant proteins (q<0.05)")
print(f"DLB vs AD: {(diff_dlb_ad['qval'] < 0.05).sum()} significant proteins (q<0.05)")
diff_dlb_cn.head(10)

In [ ]:
plot_volcano(diff_dlb_cn, title="DLB vs CN", out_dir=FIG_DIR)
plot_volcano(diff_dlb_ad, title="DLB vs AD", out_dir=FIG_DIR)

# Display inline
from IPython.display import Image, display
display(Image(FIG_DIR / "volcano_DLB_vs_CN.png"))

## 4b · Per-Protein AUC Analysis (Improvement)

Computes a standalone covariate-adjusted AUROC for **each of the 664 proteins** individually.
This extends the paper's DDC spotlight to the full proteome — revealing which proteins
carry independent discriminative value beyond the 7-protein panel.

> **Can be run independently** — does not require Section 5 (the slow CV step).

In [ ]:
import json
from lewy.protein_auc import compute_protein_aucs
from lewy.plots import plot_protein_aucs
from IPython.display import Image, display

# Runs in ~2-3 minutes on Colab CPU
auc_cn = compute_protein_aucs(X, meta, "CN", "DLB")
auc_ad = compute_protein_aucs(X, meta, "AD", "DLB")

# Save metrics
MET_DIR.mkdir(parents=True, exist_ok=True)
MET_DIR.joinpath("protein_aucs_DLB_vs_CN.json").write_text(
    json.dumps({"proteins": auc_cn.to_dict(orient="records")}, indent=2)
)
MET_DIR.joinpath("protein_aucs_DLB_vs_AD.json").write_text(
    json.dumps({"proteins": auc_ad.to_dict(orient="records")}, indent=2)
)

print(f"Ranked {len(auc_cn)} proteins (DLB vs CN), {len(auc_ad)} proteins (DLB vs AD)")

In [ ]:
print("TOP 10 — DLB vs CN")
for i, row in auc_cn.head(10).iterrows():
    marker = " *" if row["in_panel"] else ""
    print(f"  {i+1:2d}. {row['protein']:12s}  AUC={row['auc']:.3f}{marker}")

print("\nTOP 10 — DLB vs AD")
for i, row in auc_ad.head(10).iterrows():
    marker = " *" if row["in_panel"] else ""
    print(f"  {i+1:2d}. {row['protein']:12s}  AUC={row['auc']:.3f}{marker}")

print("\n(* = paper panel protein)")

In [ ]:
plot_protein_aucs(auc_cn, auc_ad, out_dir=FIG_DIR, top_n=40)
display(Image(str(FIG_DIR / "protein_aucs.png")))

## 5 · Elastic Net Classifiers (Discovery Cohort)

In [ ]:
discovery_aucs = {}
trained_clfs = {}

for pair, (pos, neg) in [("DLB_vs_CN", ("DLB", "CN")), ("DLB_vs_AD", ("DLB", "AD"))]:
    print(f"\n── {pair} ──")
    y = encode_labels(meta["Dx_group"], pos, neg).dropna().astype(int)
    X_sub = X.loc[y.index]

    cov = meta.loc[y.index, ["Age", "Sex"]].copy()
    cov["Sex_num"] = (cov["Sex"] == "Male").astype(float)
    X_with_cov = pd.concat(
        [X_sub.reset_index(drop=True), cov[["Age", "Sex_num"]].reset_index(drop=True)],
        axis=1,
    )

    clf = build_classifier()
    cache = (MET_DIR / f"cv_aucs_{pair}.json") if USE_CACHE else None
    aucs = repeated_stratified_cv(
        clf, X_with_cov, y, n_repeats=N_REPEATS, cache_path=cache, desc=pair
    )
    mean_auc, lo, hi = compute_auc_ci(aucs)
    print(f"AUC = {mean_auc:.3f}  95% CI [{lo:.3f} – {hi:.3f}]  (n={len(aucs)} folds)")

    clf.fit(X_with_cov.values, y.values)
    fpr, tpr, auc_full = roc_curve_data(clf, X_with_cov, y)
    panel = select_panel(clf, X_with_cov.columns.tolist(), max_features=7)
    print(f"Panel: {', '.join(panel)}")

    trained_clfs[pair] = (clf, X_with_cov.columns.tolist())
    discovery_aucs[pair] = {
        "mean_auc": mean_auc, "ci_lower": lo, "ci_upper": hi,
        "n_folds": len(aucs), "full_fit_auc": auc_full,
        "panel": panel,
        "roc": {"fpr": fpr.tolist(), "tpr": tpr.tolist()},
    }
    plot_roc({pair: (fpr, tpr, auc_full)}, title=pair.replace("_", " "), out_dir=FIG_DIR)

with open(MET_DIR / "discovery_aucs.json", "w") as f:
    json.dump(format_metrics(discovery_aucs), f, indent=2)

In [ ]:
display(Image(FIG_DIR / "roc_DLB_vs_CN.png"))
display(Image(FIG_DIR / "roc_DLB_vs_AD.png"))

## 6 · Panel Protein Distributions

In [ ]:
panel_present = [p for p in PANEL_PROTEINS if p in X.columns]
plot_violin(X, meta, panel_present, out_dir=FIG_DIR)
display(Image(FIG_DIR / "violin_panel_proteins.png"))

## 7 · External Validation Cohorts

In [ ]:
val_proteins = [p for p in PANEL_PROTEINS if p != "FCER2"]
val_aucs = {}
roc_data_val = {}

for pair, (pos, neg) in [("DLB_vs_CN", ("DLB", "CN")), ("DLB_vs_AD", ("DLB", "AD"))]:
    print(f"\n── {pair} ──")
    y = encode_labels(meta["Dx_group"], pos, neg).dropna().astype(int)
    X_panel = X.loc[y.index, [p for p in val_proteins if p in X.columns]]
    cov = meta.loc[y.index, ["Age", "Sex"]].copy()
    cov["Sex_num"] = (cov["Sex"] == "Male").astype(float)
    X_train = pd.concat(
        [X_panel.reset_index(drop=True), cov[["Age", "Sex_num"]].reset_index(drop=True)],
        axis=1,
    )
    clf_panel = build_classifier()
    clf_panel.fit(X_train.values, y.values)

    val_aucs[pair] = {}
    roc_data_val[pair] = {}
    for cohort_name, cohort_id in [("validation_1", 1), ("validation_2", 2), ("autopsy", "autopsy")]:
        try:
            X_val, meta_val = load_validation(cohort_id)
            result = validate_on_cohort(clf_panel, X_val, meta_val, pos, neg)
            val_aucs[pair][cohort_name] = result
            roc_data_val[pair][cohort_name] = (
                np.array(result["fpr"]), np.array(result["tpr"]), result["auc"]
            )
            print(f"  {cohort_name}: AUC={result['auc']:.3f}  (n={result['n_samples']})")
        except Exception as e:
            print(f"  {cohort_name}: SKIPPED ({e})")

with open(MET_DIR / "validation_aucs.json", "w") as f:
    json.dump(format_metrics(val_aucs), f, indent=2)

In [ ]:
# Forest plot
forest_data = {}
for pair in ("DLB_vs_CN", "DLB_vs_AD"):
    for cohort_name in ("validation_1", "validation_2", "autopsy"):
        entry = val_aucs.get(pair, {}).get(cohort_name)
        if entry:
            label = f"{pair.replace('_', ' ')} / {cohort_name}"
            auc = entry["auc"]
            forest_data[label] = (auc, auc - 0.05, auc + 0.05)

if forest_data:
    plot_forest(forest_data, title="Validation cohort AUCs", out_dir=FIG_DIR)
    display(Image(FIG_DIR / "forest_Validation_cohort_AUCs.png"))

## 8 · Summary

In [ ]:
print("=" * 55)
print("DISCOVERY RESULTS")
print("=" * 55)
for pair, vals in discovery_aucs.items():
    print(f"  {pair:20s}  AUC={vals['mean_auc']:.3f} [{vals['ci_lower']:.3f}–{vals['ci_upper']:.3f}]")
    print(f"  {'':20s}  Panel: {', '.join(vals['panel'])}")

print()
print("=" * 55)
print("VALIDATION RESULTS")
print("=" * 55)
for pair, cohorts in val_aucs.items():
    for cohort, res in cohorts.items():
        print(f"  {pair:20s}  {cohort:15s}  AUC={res['auc']:.3f}")